# FoodMacros - Nutrition5k RGB Mass Regression (Kaggle / Colab)

This is the single executable notebook for the **primary weight-estimation approach**:
RGB image -> MobileNetV2 -> food mass in grams, trained on Nutrition5k.

It uses the existing `bhumi` branch source code and does **not** retrain the classifier or YOLO detector.

## Kaggle (recommended)

1. Upload `nutrition5k.zip` as a **Kaggle Dataset** with `metadata/` and `imagery/` at the archive root.
2. Attach that dataset to this notebook using **Add Input**.
3. In **Settings**, set **Accelerator -> GPU T4 x2** (or **GPU P100** if T4 is unavailable).
4. In **Settings**, set **Internet -> On** (needed for the GitHub clone and MobileNetV2 ImageNet weights).
5. Run the cells in order, top to bottom.

Kaggle automatically extracts uploaded zip datasets under `/kaggle/input`. The notebook searches
that location, so you do not need to unzip anything manually. The 3 GB dataset is within Kaggle's
200 GB dataset limit.

## Colab (still supported)

1. Zip your local Nutrition5k folder and upload it to `MyDrive/nutrition5k.zip`.
2. Select **Runtime -> Change runtime type -> GPU (T4)**.
3. Run the cells in order, top to bottom.

The dataset has only 3,260 valid image + mass pairs because 1,506 metadata dishes have no matching
image and 240 rows have zero/invalid mass. All metrics produced here describe that verified subset;
no metric values are pre-filled or invented.

The final optional end-to-end cell is skipped automatically if the classifier artifact or Git LFS
files are unavailable. It does not affect weight-regression training.


In [ ]:
import os
import platform
import shutil
import subprocess
import sys
import tensorflow as tf

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)

print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Kaggle run type:", os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "not Kaggle"))


In [ ]:
import os
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB = False

if ON_KAGGLE:
    WORK_ROOT = Path("/kaggle/working")
    INPUT_ROOT = Path("/kaggle/input")
    REPO_DIR = WORK_ROOT / "FoodMacros"
    RESULTS_DIR = WORK_ROOT / "FoodMacros_weight"
    print("Environment: Kaggle")
elif Path("/content").exists():
    WORK_ROOT = Path("/content")
    INPUT_ROOT = Path("/content")
    REPO_DIR = WORK_ROOT / "FoodMacros"
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        ON_COLAB = True
        RESULTS_DIR = Path("/content/drive/MyDrive/FoodMacros_weight")
        print("Environment: Google Colab")
    except Exception as exc:
        RESULTS_DIR = WORK_ROOT / "FoodMacros_weight"
        print("Environment: Colab-like, but Google Drive mount failed:", exc)
else:
    WORK_ROOT = Path.cwd()
    INPUT_ROOT = WORK_ROOT
    REPO_DIR = WORK_ROOT / "FoodMacros"
    RESULTS_DIR = WORK_ROOT / "FoodMacros_weight"
    print("Environment: local/unknown")

REPO_URL = "https://github.com/Biswasrita/FoodMacros.git"
BRANCH = "bhumi"
MODEL_DIR = REPO_DIR / "models" / "weight"
ARTIFACTS_DIR = REPO_DIR / "artifacts" / "weight_dataset"
EVAL_DIR = REPO_DIR / "artifacts" / "weight_evaluation"
CLASSIFIER_DIR = REPO_DIR / "models" / "classifier"

print("WORK_ROOT:", WORK_ROOT)
print("REPO_DIR:", REPO_DIR)
print("RESULTS_DIR:", RESULTS_DIR)


In [ ]:
import os
import shutil
import subprocess

os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"

if not (REPO_DIR / "src" / "foodmacros").is_dir():
    if shutil.which("git") is None:
        raise RuntimeError(
            "git is not available. Enable Kaggle internet or add the FoodMacros source as a Kaggle dataset."
        )
    env = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1")
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO_DIR)],
        env=env,
        check=True,
    )
else:
    print("Repository already present:", REPO_DIR)

os.chdir(REPO_DIR)

if shutil.which("git-lfs") is None and ON_COLAB:
    subprocess.run(["apt-get", "-qq", "install", "-y", "git-lfs"], check=False)

if shutil.which("git-lfs") is not None:
    subprocess.run(["git", "lfs", "install"], check=False)
    lfs = subprocess.run(["git", "lfs", "pull"], check=False)
    if lfs.returncode != 0:
        print("LFS pull failed; optional classification/nutrition cell will be skipped.")
else:
    print("git-lfs not installed; optional classifier/nutrition cell will be skipped.")


In [ ]:
import importlib
import subprocess
import sys

sys.path.insert(0, str(REPO_DIR / "src"))
print("Source path:", REPO_DIR / "src")

for name in ["tensorflow", "numpy", "pandas", "sklearn", "PIL", "matplotlib"]:
    try:
        module = importlib.import_module(name)
        print(f"{name}: {getattr(module, '__version__', 'OK')}")
    except Exception as exc:
        print(f"{name}: MISSING ({exc})")

try:
    import sklearn  # noqa: F401
except Exception:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "scikit-learn"],
        check=True,
    )

import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))


In [ ]:
import zipfile
from pathlib import Path

LOCAL_ROOT = WORK_ROOT / "nutrition5k"
PROJECT_DATA_ROOT = REPO_DIR / "data" / "nutrition5k"
DRIVE_ROOT = Path("/content/drive/MyDrive")


def looks_like_nutrition5k(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "metadata").is_dir()
        and (path / "imagery").is_dir()
        and (path / "metadata" / "dish_metadata_cafe1.csv").exists()
    )


direct_candidates = [
    LOCAL_ROOT,
    PROJECT_DATA_ROOT,
    DRIVE_ROOT / "nutrition5k",
    DRIVE_ROOT / "nutrition5k_dataset",
]
if ON_KAGGLE:
    direct_candidates = [
        INPUT_ROOT / "nutrition5k" / "nutrition5k",
        INPUT_ROOT / "nutrition5k_dataset" / "nutrition5k",
        INPUT_ROOT / "nutrition5k",
        INPUT_ROOT / "nutrition5k_dataset",
    ] + direct_candidates

NUTRITION5K_ROOT = next(
    (path for path in direct_candidates if looks_like_nutrition5k(path)), None
)

if NUTRITION5K_ROOT is None and ON_KAGGLE:
    # Do not rely on a fixed nesting depth: locate the metadata file directly.
    for hit in INPUT_ROOT.rglob("dish_metadata_cafe1.csv"):
        candidate = hit.parent.parent
        if looks_like_nutrition5k(candidate):
            NUTRITION5K_ROOT = candidate
            break

if NUTRITION5K_ROOT is None:
    zip_candidates = []
    if ON_KAGGLE:
        for pattern in [
            "*nutrition5k*.zip",
            "*/*nutrition5k*.zip",
            "*/*/*nutrition5k*.zip",
            "*/*/*/*nutrition5k*.zip",
        ]:
            zip_candidates.extend(INPUT_ROOT.glob(pattern))
    else:
        zip_candidates.extend(
            [
                DRIVE_ROOT / "nutrition5k.zip",
                DRIVE_ROOT / "nutrition5k_dataset.zip",
            ]
        )
        zip_candidates.extend(sorted(DRIVE_ROOT.glob("*nutrition5k*.zip")))

    zip_path = next((path for path in zip_candidates if path.exists()), None)
    if zip_path is not None:
        print("Extracting:", zip_path)
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(WORK_ROOT)
        for path in [
            LOCAL_ROOT,
            WORK_ROOT / "nutrition5k" / "nutrition5k",
            WORK_ROOT / "nutrition5k_dataset",
            WORK_ROOT / "nutrition5k_dataset" / "nutrition5k",
        ]:
            if looks_like_nutrition5k(path):
                NUTRITION5K_ROOT = path
                break

if NUTRITION5K_ROOT is None:
    for hit in WORK_ROOT.rglob("dish_metadata_cafe1.csv"):
        candidate = hit.parent.parent
        if looks_like_nutrition5k(candidate):
            NUTRITION5K_ROOT = candidate
            break

if NUTRITION5K_ROOT is None or not looks_like_nutrition5k(NUTRITION5K_ROOT):
    raise FileNotFoundError(
        "Nutrition5k not found. On Kaggle, attach a dataset with this layout: "
        "/kaggle/input/<dataset>/nutrition5k/{metadata,imagery}. "
        "On Colab, upload nutrition5k.zip to MyDrive/nutrition5k.zip."
    )

METADATA_DIR = NUTRITION5K_ROOT / "metadata"
IMAGERY_DIR = NUTRITION5K_ROOT / "imagery"

print("NUTRITION5K_ROOT =", NUTRITION5K_ROOT)
print("Metadata files:", sorted(p.name for p in METADATA_DIR.glob("dish_metadata*.csv")))
print("Imagery exists:", IMAGERY_DIR.is_dir())

sample_rgb = next(iter(IMAGERY_DIR.glob("realsense_overhead/*/rgb.png")), None)
if sample_rgb is None:
    sample_rgb = next(
        iter(IMAGERY_DIR.glob("realsense_overhead/realsense_overhead/*/rgb.png")),
        None,
    )
if sample_rgb is None:
    raise FileNotFoundError(
        "No rgb.png found under imagery/realsense_overhead or "
        "imagery/realsense_overhead/realsense_overhead."
    )
print("Sample RGB:", sample_rgb)


In [ ]:
from foodmacros.weight.mass_dataset import build_weight_dataset

df, stats = build_weight_dataset(
    str(NUTRITION5K_ROOT),
    output_csv=ARTIFACTS_DIR / "all_pairs.csv",
)

print("Nutrition5k weight dataset")
print("--------------------------")
for key, value in stats.items():
    print(f"{key}: {value}")

assert len(df) > 0, "No valid image + mass pairs were found."


In [ ]:
from foodmacros.weight.mass_dataset import split_by_dish_id

train_df, val_df, test_df = split_by_dish_id(df, 0.7, 0.15, 0.15, seed=42)

for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(
        f"{name}: dishes={part['dish_id'].nunique()}, images={len(part)}, "
        f"mass mean={part['mass_g'].mean():.2f} g"
    )


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

sample = train_df.sample(min(9, len(train_df)), random_state=42)
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for axis, (_, row) in zip(axes.ravel(), sample.iterrows()):
    axis.imshow(Image.open(row["image_path"]).convert("RGB"))
    axis.set_title(f"{row['dish_id']}\n{row['mass_g']:.1f} g")
    axis.axis("off")
for axis in axes.ravel()[len(sample):]:
    axis.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(f"\n{name}: n={len(part)}, dishes={part['dish_id'].nunique()}")
    print(part["mass_g"].describe())


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "scripts/train_weight.py",
    "--nutrition5k", str(NUTRITION5K_ROOT),
    "--output", str(MODEL_DIR),
    "--artifacts", str(ARTIFACTS_DIR),
    "--eval-plots", str(EVAL_DIR),
    "--epochs", "40",
    "--batch-size", "16",
    "--img-size", "224",
    "--fine-tune",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True, cwd=str(REPO_DIR))


In [ ]:
from IPython.display import Image, display

curve_path = EVAL_DIR / "training_curves.png"
if curve_path.exists():
    display(Image(filename=str(curve_path)))
else:
    print("Missing:", curve_path)


In [ ]:
for name in [
    "actual_vs_predicted.png",
    "error_distribution.png",
    "mae_by_range.png",
]:
    path = EVAL_DIR / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))
    else:
        print("Missing:", path)


In [ ]:
import json

metrics_path = MODEL_DIR / "test_metrics.json"
if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2))
else:
    print("Missing:", metrics_path)


In [ ]:
import json

import pandas as pd
import tensorflow as tf
from PIL import Image

from foodmacros.data.preprocessing import load_image
from foodmacros.weight.regression import (
    get_regression_preprocess_input,
    predict_mass_only_from_array,
)

norm = json.loads((MODEL_DIR / "normalization.json").read_text())
model = tf.keras.models.load_model(MODEL_DIR / "mass_regressor.keras", compile=False)
preprocess_fn = get_regression_preprocess_input(norm.get("backbone", "mobilenetv2"))
img_size = int(norm.get("img_size", 224))

rows = []
for _, row in test_df.sample(min(9, len(test_df)), random_state=7).iterrows():
    image = load_image(row["image_path"], img_size)
    predicted = predict_mass_only_from_array(
        model, preprocess_fn, image, img_size, norm["mean"], norm["std"]
    )
    rows.append(
        {
            "image_path": row["image_path"],
            "dish_id": row["dish_id"],
            "actual_g": float(row["mass_g"]),
            "predicted_g": float(predicted),
            "abs_error_g": abs(float(predicted) - float(row["mass_g"])),
        }
    )

preview = pd.DataFrame(rows)
display(preview)

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for axis, (_, row) in zip(axes.ravel(), preview.iterrows()):
    axis.imshow(Image.open(row["image_path"]).convert("RGB"))
    axis.set_title(f"actual {row['actual_g']:.0f} g\npred {row['predicted_g']:.0f} g")
    axis.axis("off")
for axis in axes.ravel()[len(preview):]:
    axis.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
import subprocess
import sys

sample_path = test_df.iloc[0]["image_path"]
subprocess.run(
    [
        sys.executable,
        "scripts/predict_weight.py",
        "--image", str(sample_path),
        "--model", str(MODEL_DIR),
    ],
    check=True,
    cwd=str(REPO_DIR),
)


In [ ]:
import subprocess
import sys

classifier_path = CLASSIFIER_DIR / "classifier.keras"
if classifier_path.exists() and classifier_path.stat().st_size > 1_000_000:
    subprocess.run(
        [
            sys.executable,
            "scripts/run_analysis.py",
            str(sample_path),
            "--regressor-dir", str(MODEL_DIR),
        ],
        check=False,
        cwd=str(REPO_DIR),
    )
else:
    print(
        "Optional end-to-end demo skipped: classifier artifact is missing or is a Git LFS pointer."
    )


In [ ]:
import shutil

DEST = RESULTS_DIR
DEST.mkdir(parents=True, exist_ok=True)

copy_map = {
    MODEL_DIR: DEST / "models_weight",
    EVAL_DIR: DEST / "weight_evaluation",
    ARTIFACTS_DIR: DEST / "weight_dataset",
}
for source, destination in copy_map.items():
    if source.exists():
        shutil.copytree(source, destination, dirs_exist_ok=True)
        print("Copied:", source, "->", destination)
    else:
        print("Missing (skipped):", source)

archive_base = DEST.parent / "FoodMacros_weight_results"
archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=str(DEST.parent),
    base_dir=DEST.name,
)
print("Saved results to:", DEST)
print("Download archive:", archive_path)


## Outputs to collect for your submission

On Kaggle, everything is saved under `/kaggle/working/` and appears in the notebook's **Output** pane:

- `FoodMacros_weight/models_weight/mass_regressor.keras` - trained RGB weight model
- `FoodMacros_weight/models_weight/normalization.json` - mass normalization statistics
- `FoodMacros_weight/models_weight/test_metrics.json` - actual test MAE / RMSE / R2 / MAPE and baseline
- `FoodMacros_weight/weight_evaluation/training_curves.png` - training and validation loss/MAE
- `FoodMacros_weight/weight_evaluation/actual_vs_predicted.png`
- `FoodMacros_weight/weight_evaluation/error_distribution.png`
- `FoodMacros_weight/weight_evaluation/mae_by_range.png`
- `FoodMacros_weight/weight_dataset/train.csv`, `validation.csv`, `test.csv`, `all_pairs.csv`
- `FoodMacros_weight_results.zip` - one-click download of all results

On Colab, the same files are copied to `MyDrive/FoodMacros_weight/`.

The prediction cell prints a table of actual vs predicted grams for nine test dishes, and the CLI
cell prints `Estimated food weight: ... g` for one test image. These are actual model outputs for
the verified 3,260-pair subset, not pre-filled results.
